# Dependencies & Benchmarking Function

In [1]:
import numpy as np
import pandas as pd

from numba import njit, prange

In [2]:
# Setting number of logical cores in this instance
import os
os.environ["NUMBA_NUM_THREADS"] = "8"
os.environ["NUMBA_THREADING_LAYER"] = "tbb"

# Checking
print(os.environ["NUMBA_NUM_THREADS"])
print(os.environ["NUMBA_THREADING_LAYER"])

# Alternatively, through magic commands
# %env NUMBA_NUM_THREADS=8
# %env NUMBA_THREADING_LAYER=tbb  # Uses Intel's Threading Building Blocks (TBB) for threading (instead of OpenMP or workqueue).

8
tbb


In [3]:
# Benchmarking function, warmup=True skips initialization overhead
def timeit(fn, *args, warmup=True, repeat=3, **kwargs):
    """
    Run fn(*args, **kwargs) with an optional warmup, return best time and result.
    Use this to see steady-state performance after JIT compilation.
    """
    from time import perf_counter
    if warmup:
        _ = fn(*args, **kwargs)
        
    history = []
    best = float("inf")
    out = None
    for _ in range(repeat):
        t0 = perf_counter()
        out = fn(*args, **kwargs)
        dt = perf_counter()  - t0
        history.append(dt)
        best = min(best, dt)
    avg = sum(history) / repeat
    
    print(f'Best: {best:g}s\nAvg: {avg:g}s\nOutput: {out}')
    return best, avg, out

np.__version__

'2.2.6'

# Minimum Image Convention

In a system with periodic boundary conditions, like a simulation box in molecular dynamics in physics, the space "wraps around".

Suppose we want to compute the displacement vector between two points $x_1 = 0.2$ and $x_2 = 0.9$. Naively: $\Delta x = 0.7$. But this is not the shortest path, it's only 0.3 if we go backwards.

Minimum Image Convention redefines the displacement to be the shortest path on the periodic domain, ensuring $Δx∈[−L/2,+L/2].$

In [4]:
def minimum_image(dx: float, box_length: float) -> float:
    """Returns the minimum image of a coordinate difference."""
    reciprocal_half_box = 1.0 / (0.5 * box_length)  # Multiply Δx by this value to get how many "half-boxes" it spans.
    k = int(dx * reciprocal_half_box)  # k will only be 0, 1 for a 1D box.
    return dx - k * box_length  # Subtract k⋅L to pull it into the interval [−L/2,L/2]

# ---- JIT-compiled version ----
@njit('float64(float64, float64, float64)', cache=True)
def minimum_image_jit(dx, box_length, reciprocal_half_box):
    k = int(dx * reciprocal_half_box)
    return dx - k * box_length

In [ ]:
# Parameters
dx = 2
box_length = 5
reciprocal_half_box = 1.0 / (0.5 * box_length) 

print('\n------ Output (no JIT) --------\n')
for n in range(3):
    print(f"n = {n}, minimum_image({n}, 5) = {minimum_image(n, 5)}")
    
print('\n------ Output (with JIT) --------\n')
for n in range(3):
    print(f"n = {n}, minimum_image_jit({n}, 5) = {minimum_image_jit(n, box_length, reciprocal_half_box)}")



------ Output (no JIT) --------

n = 0, minimum_image(0, 5) = 0
n = 1, minimum_image(1, 5) = 1
n = 2, minimum_image(2, 5) = 2

------ Output (with JIT) --------

n = 0, minimum_image_jit_scalar(0, 5) = 0.0
n = 1, minimum_image_jit_scalar(1, 5) = 1.0
n = 2, minimum_image_jit_scalar(2, 5) = 2.0


In [14]:
# Benchmarking
print('\n------ Time (no JIT) --------\n')
best, avg, out= timeit(minimum_image, dx, box_length, repeat=30)

print('\n------ Time (with JIT) --------\n')
best_jit, avg_jit, out_jit= timeit(minimum_image_jit, dx, box_length, reciprocal_half_box, repeat=30)

print('\n------ Speed-up % --------\n')
print(f'Speed-up: {100 * (avg - avg_jit) / avg_jit:.2f}%')


------ Time (no JIT) --------

Best: 1.99972e-07s
Avg: 2.86667e-07s
Output: -0.30000000000000004

------ Time (with JIT) --------

Best: 1.00001e-07s
Avg: 2.03334e-07s
Output: -0.30000000000000004

------ Speed-up % --------

Speed-up: 40.98%


Once we know the shortest displacement Δ𝑥, we want to go halfway along that path starting from 𝑝1.

So we compute: p_mid = p1 + Δ𝑥/2.

But due to periodicity, this result might fall outside the domain $[0, L)$. Hence the `% box_length` ensures it wraps back into the primary box.


In [7]:
def midpoint_pbc(p1: float, p2: float, box_length: float) -> float:
    """Returns the midpoint between two positions under periodic boundary conditions."""
    dx = p2 - p1
    dx_mic = minimum_image(dx, box_length)
    midpoint = (p1 + 0.5 * dx_mic) % box_length
    return midpoint

@njit('float64(float64, float64, float64, float64)')
def midpoint_pbc_jit(p1, p2, box_length, reciprocal_half_box):
    """Returns the midpoint between two positions under periodic boundary conditions."""
    dx = p2 - p1
    dx_mic = minimum_image_jit(dx, box_length, reciprocal_half_box)
    midpoint = (p1 + 0.5 * dx_mic) % box_length
    return midpoint


In [8]:
# Parameters
dx = 0.7
box_length = 1
reciprocal_half_box = 1.0 / (0.5 * box_length) 

print(minimum_image(dx, box_length))
print(minimum_image_jit(dx, box_length, reciprocal_half_box))

-0.30000000000000004
-0.30000000000000004


In [9]:
# Parameters
box_length = 1.0
reciprocal_half_box = 1.0 / (0.5 * box_length) 
p1 = 0.2
p2 = 0.9

print('\n------ Output (no JIT) --------\n')
mid = midpoint_pbc(p1, p2, box_length)
print("Midpoint under PBC:", mid)  # Expected: 0.05

print('\n------ Output (with JIT) --------\n')
mid_jit = midpoint_pbc_jit(p1, p2, box_length, reciprocal_half_box)
print("Midpoint under PBC (JIT):", mid_jit)  # Expected: 0.05


------ Output (no JIT) --------

Midpoint under PBC: 0.04999999999999999

------ Output (with JIT) --------

Midpoint under PBC (JIT): 0.04999999999999999


In [10]:
# Benchmarking
print('\n------ Time (no JIT) --------\n')
best, avg, out= timeit(minimum_image, dx, box_length, repeat=30)

print('\n------ Time (with JIT) --------\n')
best_jit, avg_jit, out_jit= timeit(minimum_image_jit, dx, box_length, reciprocal_half_box, repeat=30)

print('\n------ Speed-up % --------\n')
print(f'Speed-up: {100 * (avg - avg_jit) / avg_jit:.2f}%')


------ Time (no JIT) --------

Best: 2.99973e-07s
Avg: 5.33333e-07s
Output: -0.30000000000000004

------ Time (with JIT) --------

Best: 2.00002e-07s
Avg: 3.46668e-07s
Output: -0.30000000000000004

------ Speed-up % --------

Speed-up: 53.85%


## Centroid under PBC:  Mapping to the unit circle

We can think of each coordinate as lying on a circle of circumference L and represent each coordinate value as an angle:
$$
\theta = \frac{2\pi x}{L}
$$
Then:
1. Convert each coordinate to $(\cos{\theta},\sin{\theta})$
2. Average the cosines and sines over all particles
3. Get the average angle via $\text{atan2} (\overline{\sin{\theta}}, \overline{\cos{\theta}})$
4. Map that angle back to $[0,L)$ to get the PBC-aware centroid coordinate.

In [10]:
def centroid_pbc(positions, box_length):
    """
    Computes the centroid of n particles in 3D under periodic boundary conditions.
    
    Parameters:
    - positions: numpy array of shape (n, 3), coordinates of particles
    - box_length: float, box size (assumes cubic box)
    
    Returns:
    - centroid: numpy array of shape (3,), the PBC-aware centroid
    """
    centroid = np.zeros(3)
    for dim in range(3):  # process x, y, z separately
        theta = 2 * np.pi * positions[:, dim] / box_length
        sin_sum = np.sum(np.sin(theta))
        cos_sum = np.sum(np.cos(theta))
        avg_theta = np.arctan2(sin_sum, cos_sum)
        if avg_theta < 0:  # ensure angle in [0, 2π)
            avg_theta += 2 * np.pi
        centroid[dim] = (avg_theta / (2 * np.pi)) * box_length
    return centroid

In [12]:
box_length = 1.0
positions = np.array([
    [0.2, 0.3, 0.4],
    [0.9, 0.8, 0.2],
    [0.1, 0.9, 0.7]
])

centroid = centroid_pbc(positions, box_length)
print("Centroid under PBC:", centroid)

Centroid under PBC: [0.07296583 0.9        0.4       ]
